# Statistical Drift Detection: Hypothesis Testing Across Real Data Batches

**Topic:** Statistics, biased toward ML quality engineering (drift detection, sanity-checking your own tests)

## Problem statement

In production ML systems you rarely get a clean "the model broke" signal. More often you get
two batches of data — the one the model was trained/validated on, and a newer one arriving
in production — and you need to decide, with evidence rather than eyeballing summary
statistics, whether the newer batch comes from a meaningfully different distribution. Calling
"drift" too eagerly (false alarms that page someone every night) is as costly as missing real
drift (a model silently degrading on data it was never validated against). This is exactly the
kind of signal-vs-noise judgment call an SQA background is good at: you're not just running a
test, you're deciding whether the test itself is trustworthy.

You'll use the real **UCI Wine Quality** dataset, split into its two naturally occurring
sub-populations — **red wine** (1,599 rows) and **white wine** (4,898 rows) — as a stand-in for
"reference batch" (e.g. training-time data) vs. "current batch" (e.g. this week's production
traffic). These are two real, differently-produced populations, not injected synthetic noise,
so the drift you'll detect is genuine distributional difference, not something we faked.

## What you should produce

1. `population_stability_index(reference, current, bins=10)` — a function implementing the
   PSI metric (a standard industry heuristic for feature drift: PSI < 0.1 = no meaningful
   shift, 0.1-0.2 = moderate, > 0.2 = significant).
2. `drift_report(reference_df, current_df, features, alpha=0.05)` — a function that, for each
   feature, runs a two-sample KS test and a Welch's t-test, Bonferroni-corrects the KS p-values
   across features, computes PSI, and returns a tidy `DataFrame` with one row per feature and a
   boolean `drift_flag` (flagged if the corrected KS p-value is below `alpha` **or** PSI > 0.2).
3. Run `drift_report` comparing red wine (reference) against white wine (current) and report,
   in a sentence, which features drifted and which (if any) didn't.
4. **A negative control for your own test**: randomly pool the red and white rows together and
   re-split them 50/50 (same sizes as the real red/white split) with no label information, then
   re-run `drift_report` on that random split. If your test pipeline is sound, it should report
   (close to) **no** drift here, because there is none by construction.
5. **A stretch goal**: repeat step 4 for 30 different random seeds and report what fraction of
   those 30 x 11 (trials x features) checks were flagged as drift at `alpha=0.05`. Compare that
   empirical false-positive rate to the ~5% you'd expect from the significance level alone —
   this is the statistical-testing equivalent of checking your test suite isn't flaky.


## Setup

Imports and data loading are provided below — no TODOs here, this part isn't the exercise.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

RNG_SEED = 42
np.random.seed(RNG_SEED)


In [ ]:
# Real dataset: UCI Wine Quality (red + white), mirrored as plain CSV (no header row) at
# https://github.com/jbrownlee/Datasets — identical values to the original UCI files,
# just comma-separated instead of semicolon-separated.
COLUMNS = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides",
    "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates",
    "alcohol", "quality",
]

RED_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-red.csv"
WHITE_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-white.csv"

red = pd.read_csv(RED_URL, header=None, names=COLUMNS)
white = pd.read_csv(WHITE_URL, header=None, names=COLUMNS)

FEATURES = [c for c in COLUMNS if c != "quality"]  # drift-check the chemistry features, not the label

print(f"red:   {red.shape}")
print(f"white: {white.shape}")
assert red.isna().sum().sum() == 0 and white.isna().sum().sum() == 0, "unexpected missing values"
red.head()


## Task 1 — Population Stability Index (PSI)

Fill in `population_stability_index`. Steps:
- Build `bins + 1` quantile edges **from the reference array only** (this matters: PSI compares
  the current batch against the reference's own distribution, not some global binning).
- Widen the first/last edge slightly so the min/max of either array isn't dropped by
  `np.histogram`, and de-duplicate edges (quantiles can collide on skewed features).
- Compute each bin's share of points in `reference` and in `current`.
- Clip shares away from exactly 0 (e.g. to `1e-6`) before taking a log, to avoid `log(0)`.
- Return `sum((current_pct - reference_pct) * log(current_pct / reference_pct))`.


In [ ]:
def population_stability_index(reference: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    """Return the Population Stability Index between two 1-D numeric arrays."""
    # TODO: build quantile bin edges from `reference` only
    # TODO: histogram both arrays into those edges, convert counts to proportions
    # TODO: clip proportions away from 0, return the PSI value
    raise NotImplementedError


## Task 2 — Drift report across features

Fill in `drift_report`. For each feature in `features`:
- Run `scipy.stats.ks_2samp(reference_df[f], current_df[f])` for the KS statistic and p-value.
- Run `scipy.stats.ttest_ind(reference_df[f], current_df[f], equal_var=False)` (Welch's t-test,
  since the two batches have very different sizes and we shouldn't assume equal variance).
- Run `population_stability_index` on the same two columns.
- Collect everything into one row per feature.

Then, across the whole feature table:
- Apply a **Bonferroni correction** to the KS p-values: `ks_p_adj = min(ks_p * n_features, 1.0)`.
  (Running 11 independent tests at alpha=0.05 each gives you a much higher chance that *some*
  test falsely fires than 5% — Bonferroni is the blunt, defensible-in-an-SQA-report way to
  control that.)
- Set `drift_flag = (ks_p_adj < alpha) | (psi > 0.2)` — flag on either strong statistical
  evidence or a large practical-significance PSI, since a huge sample can make a trivial shift
  "significant" and a small sample can hide a real one.


In [ ]:
def drift_report(reference_df: pd.DataFrame, current_df: pd.DataFrame, features: list[str],
                  alpha: float = 0.05) -> pd.DataFrame:
    """Return a per-feature drift report comparing reference_df against current_df."""
    # TODO: for each feature, compute ks_stat, ks_p, t_p, psi and collect into rows
    # TODO: Bonferroni-correct ks_p across len(features) into ks_p_adj
    # TODO: set drift_flag from ks_p_adj and psi, return a DataFrame sorted by ks_p_adj
    raise NotImplementedError


## Task 3 — Run it on the real split

Call `drift_report(red, white, FEATURES)` and inspect the result. Which features show the
clearest drift (lowest `ks_p_adj`, highest PSI)? Is there any feature that does **not** get
flagged? Write one sentence of interpretation below the table.


In [ ]:
# TODO: real_report = drift_report(red, white, FEATURES)
# TODO: display real_report sorted however you like, and write your one-sentence interpretation


## Task 4 — Negative control: test the test

Pool `red[FEATURES]` and `white[FEATURES]` into one `DataFrame`, then use
`np.random.default_rng(RNG_SEED).permutation(...)` to produce a random 50/50-by-size split that
matches the real `len(red)` / `len(white)` sizes but carries **no** red/white label information.
Re-run `drift_report` on that random split. You should see few or no `drift_flag == True` rows,
because there's no real distributional difference between two random halves of the same pool —
this is your check that `drift_report` isn't just flagging everything regardless of input.


In [ ]:
# TODO: pooled = pd.concat([red[FEATURES], white[FEATURES]], ignore_index=True)
# TODO: build a random permutation and split it into two groups sized len(red) and len(white)
# TODO: control_report = drift_report(shuffled_group_a, shuffled_group_b, FEATURES)
# TODO: display control_report and compare its drift_flag count against Task 3's


## Task 5 (stretch) — empirical false-positive rate

Repeat Task 4's random split for 30 different seeds (e.g. `range(30)`), tally how many of the
`30 * len(FEATURES)` individual feature checks get `drift_flag == True`, and report that as a
fraction. Compare it to the nominal `alpha = 0.05` — is the empirical rate close to 5%, well
above it, or well below it? Can you explain the direction of the gap (hint: Bonferroni corrects
the **family-wise** error rate across all 11 features at once — think about what that does to
each individual feature's effective significance threshold, versus running each test alone at
`alpha = 0.05`)?


In [ ]:
# TODO: loop over 30 seeds, each time building a fresh random split and running drift_report
# TODO: accumulate total flagged checks vs total checks run, print the empirical false-positive rate


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

```python
def population_stability_index(reference: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0] -= 1e-9
    edges[-1] += 1e-9
    edges = np.unique(edges)  # skewed features can produce duplicate quantile edges

    ref_counts, _ = np.histogram(reference, bins=edges)
    cur_counts, _ = np.histogram(current, bins=edges)

    ref_pct = np.clip(ref_counts / len(reference), 1e-6, None)
    cur_pct = np.clip(cur_counts / len(current), 1e-6, None)

    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


def drift_report(reference_df: pd.DataFrame, current_df: pd.DataFrame, features: list[str],
                  alpha: float = 0.05) -> pd.DataFrame:
    rows = []
    for f in features:
        ks_stat, ks_p = stats.ks_2samp(reference_df[f], current_df[f])
        _, t_p = stats.ttest_ind(reference_df[f], current_df[f], equal_var=False)
        psi = population_stability_index(reference_df[f].to_numpy(), current_df[f].to_numpy())
        rows.append({"feature": f, "ks_stat": ks_stat, "ks_p": ks_p, "t_p": t_p, "psi": psi})

    report = pd.DataFrame(rows)
    m = len(features)
    report["ks_p_adj"] = np.minimum(report["ks_p"] * m, 1.0)  # Bonferroni correction
    report["drift_flag"] = (report["ks_p_adj"] < alpha) | (report["psi"] > 0.2)
    return report.sort_values("ks_p_adj").reset_index(drop=True)


# --- Task 3: real red vs. white split ---------------------------------------
real_report = drift_report(red, white, FEATURES)
print(real_report.round(4))
print(f"\nflagged: {real_report['drift_flag'].sum()} / {len(FEATURES)} features")
# Every single chemistry feature is flagged. Red and white wine are made with different
# processes (e.g. skin contact, sulfite use), so this is genuine, expected drift -- not noise.

# --- Task 4: negative control (random 50/50-by-size split of the pooled rows) ------
pooled = pd.concat([red[FEATURES], white[FEATURES]], ignore_index=True)
rng = np.random.default_rng(RNG_SEED)
perm = rng.permutation(len(pooled))
split_point = len(red)
group_a = pooled.iloc[perm[:split_point]]
group_b = pooled.iloc[perm[split_point:]]

control_report = drift_report(group_a, group_b, FEATURES)
print(control_report.round(4))
print(f"\nflagged (control): {control_report['drift_flag'].sum()} / {len(FEATURES)} features")
# With no real label information behind the split, almost nothing should fire -- this is the
# sanity check that drift_report has a sane false-positive rate, not that it flags everything.

# --- Task 5: empirical false-positive rate across 30 random splits ----------------
total_checks = 0
total_flagged = 0
for seed in range(30):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(pooled))
    a = pooled.iloc[perm[:split_point]]
    b = pooled.iloc[perm[split_point:]]
    rep = drift_report(a, b, FEATURES)
    total_checks += len(rep)
    total_flagged += int(rep["drift_flag"].sum())

empirical_fpr = total_flagged / total_checks
print(f"\nempirical false-positive rate: {empirical_fpr:.3f} "
      f"({total_flagged}/{total_checks}) vs nominal alpha=0.05")
# Comes out at 0.000 here -- well *below* the nominal 5%, not above it. Bonferroni controls the
# family-wise error rate (the chance that *any* of the 11 features falsely fires) at alpha, which
# makes each individual feature's effective threshold roughly alpha / 11 ~= 0.0045 -- much
# stricter than testing one feature alone at alpha=0.05. The PSI > 0.2 clause adds essentially no
# extra false positives here because a 50/50 random split of the same pool rarely produces a PSI
# that large by chance. So the combined drift_flag is conservative: it under-fires on random
# noise, which is the safe direction for a production alert you don't want paging anyone at 2am
# for nothing -- but it's worth documenting that this conservatism also means a small, real
# drift might need several features' worth of evidence (or a higher alpha) before it fires.
```

</details>
